# LLM Fine-Tuning Pipeline using LLaMA-Factory

**Project Overview & Objectives**

Welcome to this fine-tuning walkthrough. The primary objective of this project is to establish a streamlined pipeline for fine-tuning Large Language Models (LLMs) using the LLaMA-Factory framework. Specifically, we are utilizing QLoRA (Quantized Low-Rank Adaptation) to fine-tune the Gemma-1.1-2b-it model.

**By the end of this document, you will understand how to:**

- Set up a high-performance environment for LLM training.

- Leverage both WebUI and Command Line Interface (CLI) methods for model training.

- Apply parameter-efficient fine-tuning (PEFT) techniques to train models on consumer-grade hardware.

- Perform inference to evaluate the generated text quality.

# Environment Setup

**Context Block:** The following cell clones the LLaMA-Factory repository, which provides a unified framework for LLM training. We then change our working directory and install essential dependencies. Notice the specific installation of bitsandbytes—this library is crucial because it enables 4-bit and 8-bit quantization, allowing us to load massive models into limited GPU memory without triggering Out-Of-Memory (OOM) errors. Finally, we authenticate with Hugging Face to access base models and datasets.



In [ ]:
# Clone the LLaMA-Factory repository for our training framework
!git clone https://github.com/hiyouga/LlamaFactory.git

%cd /content/LlamaFactory

In [ ]:
# Install LlamaFactory dependencies
!pip install -e .
!pip install -r requirements/metrics.txt

# Install bitsandbytes for model quantization (QLoRA requirement)
!pip install bitsandbytes>=0.39.0

In [ ]:
import os
from llamafactory.webui.interface import create_ui

In [ ]:
# Configure Gradio to generate public shareable links for the WebUI
os.environ["GRADIO_SHARE"] = "1"

In [ ]:
# For better security, store your Hugging Face token in Colab's secrets manager.
from google.colab import userdata
os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN") # Use the name you give your secret

# Authenticate with Hugging Face (Requires a valid token)
!hf auth login --token "$HF_TOKEN"

# Data Acquisition

Add the below code into /content/LlamaFactory/data/dataset_info.json for adding custom datasets:
```
"hf_dataset":{
    "hf_hub_url":"harpomaxx/unix-commands",
    "columns":{
      "prompt":"instruction",
      "query":"input",
      "response":"output"
    }
  }
```

# Methods to launch LlamaFactory

## METHOD 1: WebUI Launch for GUI-based configuration

In [ ]:
ui = create_ui()
ui.launch(share=True)

## METHOD 2: CLI-Based Training Execution

In [ ]:
# To avoid CUDA blocking errors during complex memory operations
%env CUDA_LAUNCH_BLOCKING=1

# Execute training via LLaMA-Factory CLI utilizing a YAML configuration file.
# Note: The equivalent raw CLI command looks like this:
# !python -m llamafactory.cli train \
#   --model_name_or_path google/gemma-1.1-2b-it \
#   --template gemma \
#   --stage sft \
#   --finetuning_type lora \
#   --dataset yahma/alpaca-cleaned \
#   --output_dir output/my-gemma-qlora \
#   --cutoff_len 2048 \
#   --per_device_train_batch_size 1 \
#   --gradient_accumulation_steps 8 \
#   --num_train_epochs 1 \
#   --learning_rate 5e-5 \
#   --lora_rank 64 \
#   --quantization_bit 4 \
#   --fp16 True

!python -m llamafactory.cli train train_gemma_qlora.yaml

# Evaluation & Results

**Context Block:** This cell orchestrates the inference phase. We instantiate the tokenizer and the base model using the transformers library, ensuring the data types are optimized (torch.float16). We then wrap the base model with PeftModel to dynamically apply our trained LoRA weights. Finally, we pass a test prompt to observe the model's generation capabilities.

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel
import torch

In [ ]:
# Define model paths
base_model_path = "google/gemma-1.1-2b-it"
# Path to your trained LoRA adapter (adjust as needed based on output dir)
adapter_path = "/content/LlamaFactory/gemma_lora_sft_output"

# 1. Initialize the Tokenizer
tokenizer = AutoTokenizer.from_pretrained(base_model_path)

# 2. Load the Base Model in half-precision for inference speed
base_model = AutoModelForCausalLM.from_pretrained(
    base_model_path,
    device_map="auto",
    dtype=torch.float16
)

# 3. Apply the LoRA adapter to the base model
model = PeftModel.from_pretrained(base_model, adapter_path)
model.eval() # Set model to evaluation mode

# 4. Define the inference prompt
prompt = "Explain what is QLoRA"

# 5. Tokenize input and move to the appropriate hardware (GPU)
inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

# 6. Generate the response
# Using max_new_tokens to prevent runaway generation
outputs = model.generate(
    **inputs,
    max_new_tokens=200
)

# 7. Decode the generated tokens back into human-readable text
response_text = tokenizer.decode(outputs[0], skip_special_tokens=True)
print(response_text)